In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import torch
from torch.utils.data import DataLoader

from dataset import SceneTwoPairsDataset
from model import PairImageCylinderModel
from losses import supervised_loss


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA available: True
NVIDIA GeForce RTX 5050 Laptop GPU


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = PairImageCylinderModel(
    img_size=128,
    patch_size=16,
    in_chans=3,
    embed_dim=256,
    depth=4,
    num_heads=4,
    num_bins=32,
    dropout=0.1,
).to(device)

In [5]:
from torch.utils.data import DataLoader

dataset = SceneTwoPairsDataset(
    root_dir="dataset",
    image_size=128,
    scale_xy=2.8,
    return_paths=False,
)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
)

In [6]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

model.train()
for epoch in range(20):
    total = 0.0

    for batch in loader:
        img_a, vision_a, img_b, vision_b, pose_ab = batch

        img_a = img_a.to(device, non_blocking=True)
        img_b = img_b.to(device, non_blocking=True)
        vision_a = vision_a.to(device, non_blocking=True)
        pose_ab = pose_ab.to(device, non_blocking=True)

        pred_vision, pred_pose = model(img_a, img_b)

        loss, loss_vis, loss_pose = supervised_loss(
            pred_vision, vision_a,
            pred_pose, pose_ab,
            lambda_pose=1.0,
        )

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {epoch+1}: loss={total / len(loader):.4f}")

Epoch 1: loss=2.9509
Epoch 2: loss=2.1322
Epoch 3: loss=2.3428
Epoch 4: loss=2.1759
Epoch 5: loss=2.4312
Epoch 6: loss=2.1327
Epoch 7: loss=1.7782
Epoch 8: loss=2.1762
Epoch 9: loss=1.9482
Epoch 10: loss=2.0864
Epoch 11: loss=2.1357
Epoch 12: loss=2.2255
Epoch 13: loss=2.0588
Epoch 14: loss=2.0917
Epoch 15: loss=1.9842
Epoch 16: loss=2.0143
Epoch 17: loss=1.9264
Epoch 18: loss=1.7237
Epoch 19: loss=1.7509
Epoch 20: loss=2.1164


In [7]:
# Cell: sanity check för två bildpar

import matplotlib.pyplot as plt

def show_two_pairs(img_a1, img_b1, img_a2, img_b2, max_items=2):
    img_a1 = img_a1.cpu()
    img_b1 = img_b1.cpu()
    img_a2 = img_a2.cpu()
    img_b2 = img_b2.cpu()

    n = min(img_a1.shape[0], max_items)

    plt.figure(figsize=(8, 4 * n))

    for i in range(n):
        # Första paret
        plt.subplot(n, 4, 4*i + 1)
        plt.imshow(img_a1[i].permute(1, 2, 0))
        plt.title("Pair1 - A")
        plt.axis("off")

        plt.subplot(n, 4, 4*i + 2)
        plt.imshow(img_b1[i].permute(1, 2, 0))
        plt.title("Pair1 - B")
        plt.axis("off")

        # Andra paret
        plt.subplot(n, 4, 4*i + 3)
        plt.imshow(img_a2[i].permute(1, 2, 0))
        plt.title("Pair2 - A")
        plt.axis("off")

        plt.subplot(n, 4, 4*i + 4)
        plt.imshow(img_b2[i].permute(1, 2, 0))
        plt.title("Pair2 - B")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


# Hämta batch
batch = next(iter(loader))
img_a1, img_b1, img_a2, img_b2 = batch

# Visa bilder
show_two_pairs(img_a1, img_b1, img_a2, img_b2)

# Skicka till device
img_a1 = img_a1.to(device)
img_b1 = img_b1.to(device)
img_a2 = img_a2.to(device)
img_b2 = img_b2.to(device)

# Kör modellen
with torch.no_grad():
    pred1 = model(img_a1, img_b1)
    pred2 = model(img_a2, img_b2)

print("Pred1 shape:", pred1.shape)  # [B, K, 5]
print("Pred2 shape:", pred2.shape)

print("\nExempel cylinder (pair1):", pred1[0, 0])
print("Exempel cylinder (pair2):", pred2[0, 0])
print("Confidence (pair1):", pred1[0, :, 4])

ValueError: too many values to unpack (expected 4)